# ReFactX Knowledge-Graph Generation backed by Neo4j (multi-hop)

Same idea as `notebooks/try_refactx_knowledge_graph.ipynb`, but the three-level
API is served from a **Neo4j** property graph instead of the synthetic
rule-backed store:

1. `entity_index()` — the entity names that exist (what the prefix tree indexes);
2. `property_strings(entity)` — the relations that can answer for that entity;
3. `get(subject, relation)` — the objects that answer the relation.

The connector lives in `refactx/neo4j_kg.py` and exposes both a
`Neo4jKnowledgeGraph` class and module-level functions with the same names as the
rule-backed `api.py`, so it drops straight into `KnowledgeGraphGeneration`.

This notebook focuses on **multi-hop chains**: with `long_chains=True` the object
of a hop becomes the subject of the next, and the model may keep extending the
path (or stop) at every hop boundary, so a single `<kg>` call can return a whole
path such as `<Dreams from My Father> <BOOK_WRITTEN_WORK_AUTHOR> <Barack Obama>
<PEOPLE_PERSON_PLACE_OF_BIRTH> <Honolulu> <LOCATION_LOCATION_CONTAINEDBY> <Hawaii> .`

Graph model: nodes are `(:Entity {id, name})` and Freebase predicates are
sanitized into relationship types, e.g. `/people/person/place_of_birth` ->
`PEOPLE_PERSON_PLACE_OF_BIRTH`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path

# Make sure the local ReFactX checkout is imported (overrides any installed copy).
REPO = Path.cwd()
if not (REPO / 'refactx').is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print('Repo:', REPO)

In [ ]:
import torch
from transformers import AutoProcessor, AutoTokenizer, AutoModelForImageTextToText, TextStreamer, ProcessorMixin

import refactx
from refactx.index import DictIndex
from refactx.generate import ConstrainedLogitsProcessor, ConstrainedStateList, KnowledgeGraphGeneration
from refactx.neo4j_kg import Neo4jKnowledgeGraph

In [ ]:
MODEL = 'Qwen/Qwen3.5-4B'        # set to 'Qwen/Qwen3.5-0.8B' for a quick smoke test
# MODEL = 'Qwen/Qwen3.5-0.8B'
NUM_BEAMS = 1
NUM_BATCHES = 1
LONG_CHAINS = True               # object of a hop becomes the subject of the next
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

## Connect to Neo4j and preview the connectors

Connection parameters come from `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
(`bolt://localhost:7687`, `neo4j` / `password` by default) and `NEO4J_DATABASE`
(unset -> the server's default database).

In [ ]:
kg = Neo4jKnowledgeGraph()          # pass skip_prefixes=() to keep BASE_*/COMMON_* relations
print('connected | database:', kg.database)

# The index is seeded by expanding the neighbourhood of a few entities so the
# demo stays small. Only the *first* entity of a path must be in this index: with
# long_chains the following subjects come from the graph callbacks.
SEEDS = ['Barack Obama', 'Paris', 'Albert Einstein']
entity_index = lambda: kg.entity_index(limit=5000, seeds=SEEDS, hops=1)
property_strings = kg.property_strings
get = kg.get

In [ ]:
entities = entity_index()
print(f'{len(entities)} entities, e.g. {entities[:5]}')

# Preview a multi-hop chain: book -> author -> place of birth -> contained-by.
print('\nchain preview:')
for subj, rel in [('Dreams from My Father', 'BOOK_WRITTEN_WORK_AUTHOR'),
                  ('Barack Obama', 'PEOPLE_PERSON_PLACE_OF_BIRTH'),
                  ('Honolulu', 'LOCATION_LOCATION_CONTAINEDBY')]:
    print(f'  {subj} --{rel}--> {get(subj, rel)[:4]}')

## Load the model

In [ ]:
try:
    processor = AutoProcessor.from_pretrained(MODEL)   # Qwen3.5-4B ships a processor config
except Exception:
    processor = AutoTokenizer.from_pretrained(MODEL)   # text-only fallback (e.g. Qwen3.5-0.8B)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL, device_map='auto', dtype='bfloat16')
tokenizer = processor
streamer = TextStreamer(tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer,
                        skip_prompt=True)

## Build the KG index and register `KnowledgeGraphGeneration`

`<kg>` activates graph traversal. Each hop is serialized as
`<subject> <relation> <object>`. With `long_chains=True` the object becomes the
next subject, and at every hop boundary the model may either follow another
relation or stop the path, so one `<kg>` call can span several hops.

In [ ]:
def _tok():
    return tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer

def _encode(text):
    return _tok().encode(text, add_special_tokens=False)

index = DictIndex()
index.set_tokenizer(tokenizer)
entities = entity_index()
for entity in entities:
    # leading space: the model emits `<kg> <Subject> <relation> <object> ...`
    index.add(_encode(f' <{entity}>'))

print(f'Loaded {len(entities)} entities into the KG index.')

In [ ]:
states = ConstrainedStateList(
    'auto',
    num_beams=NUM_BEAMS,
    num_batches=NUM_BATCHES,
    debug_tokenizer=tokenizer,
)

logits_processor = ConstrainedLogitsProcessor(states=states, tokenizer=tokenizer)
logits_processor.add_pattern(
    '<kg>',
    KnowledgeGraphGeneration,
    index=index,
    get_relations=property_strings,
    get_objects=get,
    long_chains=LONG_CHAINS,
    eot=' </kg>\n',
)

In [ ]:
# Multi-hop prompt: a single <kg> call may return a whole chained path.
prompt_messages = refactx.load_prompt(str(REPO / 'prompts' / 'prompt_qwen36_angular2_kg_longchain.yaml'))
len(prompt_messages)

## Ask a multi-hop question

The system prompt asks the model to gather each chain in one `<kg>` call. Every
returned triple is collected from `generated_path_metadata`.

In [ ]:
def _tokenize(text):
    tok = _tok()
    return tok(text, return_tensors='pt').to(model.device)

def _decode(ids):
    return _tok().decode(ids, skip_special_tokens=True)

def ask(question, max_new_tokens=512, thinking=False, stream=True, **generation_kwargs):
    logits_processor.reset_states()
    prompt = refactx.apply_prompt_template(
        tokenizer, question=question, enable_thinking=thinking, prompt_template=prompt_messages)
    inputs = _tokenize(prompt)
    model.eval()
    started = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            logits_processor=[logits_processor],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.80,
            top_k=20,
            min_p=0.0,
            repetition_penalty=1.0,
            num_beams=NUM_BEAMS,
            num_return_sequences=1,
            use_cache=True,
            streamer=streamer if stream else None,
            **generation_kwargs,
        )
    state = logits_processor.states[0, 0]
    answer = _decode(output[0][inputs.input_ids.shape[1]:])
    paths = []
    for generation in state.generation_history:
        paths.extend(getattr(generation, 'generated_path_metadata', []))
    print(f'\n--- KG paths ({len(paths)}) ---')
    for path in paths:
        print(path)
    print(f'--- elapsed {time.time() - started:.2f}s ---')
    return answer, paths

In [ ]:
# A multi-hop question: the answer needs book -> author -> birthplace -> state.
question = 'Which state is the birthplace of the author of "Dreams from My Father" in?'
print('Question:', question)
answer, paths = ask(question)
print(answer)

In [ ]:
# Try more multi-hop questions (all start from an entity in the index):
# ask('Where was the author of "Dreams from My Father" born?')
# ask('Which state is the birthplace of the author of "The Audacity of Hope" in?')
# ask('Which country is the birthplace of the author of "Dreams from My Father" in?')

In [ ]:
# kg.close()